### Comparaison des Modélisations pour la Fréquence et les Coûts des Sinistres

Dans cette section, nous allons comparer différentes approches de modélisation afin d'évaluer deux aspects clés des sinistres :

1. **La fréquence des sinistres** : Modélisation du nombre de sinistres survenus pour chaque contrat.
2. **Les coûts des sinistres** : Estimation des montants associés aux sinistres.

L'objectif est d'identifier les modèles les plus performants pour chaque aspect, en utilisant des métriques d'évaluation adaptées.

In [30]:
#pip install -r requirement.txt

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import numpy as np
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance
import re
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm

import numpy as np
from sklearn.model_selection import KFold
from pyglmnet import GLM


In [2]:
freq = pd.read_parquet("data/raw/freMTPLfreq.parquet")
sev = pd.read_parquet("data/raw/freMTPLsev.parquet")

In [3]:
#Combiner les 2 bases de données

freq['PolicyID'] = freq['PolicyID'].astype(str)
sev['PolicyID'] = sev['PolicyID'].astype(str)

# Agrégation de sev : un contrat peut avoir plusieurs sinistres
sev_agg = (
    sev.groupby("PolicyID")
       .agg(
           ClaimAmount=("ClaimAmount", "sum"),   # coût total des sinistres du contrat
           ClaimNb_sev=("ClaimAmount", "size")   # nombre de sinistres déclarés dans sev
       )
       .reset_index()
)

print(len(sev), "lignes sev brutes")
print(len(sev_agg), "lignes sev agrégées (1 par PolicyID)")

merged = pd.merge(freq, sev_agg, on="PolicyID", how="left")

# Remplacer les NaN (contrats sans sinistre) par 0
merged["ClaimAmount"] = merged["ClaimAmount"].fillna(0)
merged["ClaimNb_sev"] = merged["ClaimNb_sev"].fillna(0)

print("Nb de lignes freq :", len(freq))
print("Nb de lignes merged :", len(merged))


16181 lignes sev brutes
15390 lignes sev agrégées (1 par PolicyID)
Nb de lignes freq : 413169
Nb de lignes merged : 413169


# Création de différentes classes actuarielles nécessaires pour notre modélisation


In [4]:
# 1) Contrôles de base
print(merged[["PolicyID","ClaimNb","Exposure","ClaimAmount"]].head())
print(merged[["ClaimNb","Exposure","ClaimAmount"]].describe())

# 2) Variables actuarielles de base
merged["Freq"] = merged["ClaimNb"] / merged["Exposure"]          # fréquence annuelle
merged["PurePremium"] = merged["ClaimAmount"] / merged["Exposure"]  # prime pure
merged["AvgClaim"] = np.where(                                    # coût moyen conditionnel
    merged["ClaimNb"] > 0,
    merged["ClaimAmount"] / merged["ClaimNb"],
    np.nan
)

# 3) Quelques ratios globaux (à commenter dans le rapport)
tot_expo = merged["Exposure"].sum()
tot_claims = merged["ClaimNb"].sum()
tot_amount = merged["ClaimAmount"].sum()

freq_globale = tot_claims / tot_expo
pure_prem_globale = tot_amount / tot_expo
avgclaim_globale = tot_amount / tot_claims

print("Fréquence globale :", freq_globale)
print("Prime pure globale :", pure_prem_globale)
print("Coût moyen global :", avgclaim_globale)
print(f"% contrats sans sinistre: {(merged['ClaimNb']==0).mean()*100:.1f}%")

merged.columns

  PolicyID  ClaimNb  Exposure  ClaimAmount
0        1        0      0.09          0.0
1        2        0      0.84          0.0
2        3        0      0.52          0.0
3        4        0      0.45          0.0
4        5        0      0.15          0.0
             ClaimNb       Exposure   ClaimAmount
count  413169.000000  413169.000000  4.131690e+05
mean        0.039163       0.561088  8.341642e+01
std         0.204053       0.369477  4.192526e+03
min         0.000000       0.002732  0.000000e+00
25%         0.000000       0.200000  0.000000e+00
50%         0.000000       0.540000  0.000000e+00
75%         0.000000       1.000000  0.000000e+00
max         4.000000       1.990000  2.036833e+06
Fréquence globale : 0.06979858984933181
Prime pure globale : 148.66904231188673
Coût moyen global : 2129.9720042024596
% contrats sans sinistre: 96.3%


Index(['PolicyID', 'ClaimNb', 'Exposure', 'Power', 'CarAge', 'DriverAge',
       'Brand', 'Gas', 'Region', 'Density', 'ClaimAmount', 'ClaimNb_sev',
       'Freq', 'PurePremium', 'AvgClaim'],
      dtype='object')

# Premières approches de modélisation avec loi Poisson pour la fréquence et loi Gamma pour la sévérité

In [5]:

# ============================================================
# 1. Préparation des données et création des classes actuarielles
# ============================================================

# On garde uniquement les contrats exposés
df_glm = merged[merged["Exposure"] > 0].copy()

# Variables de classes si besoin
bins_driver = [17, 25, 30, 40, 50, 60, 120]
labels_driver = ["<25", "25-29", "30-39", "40-49", "50-59", "60+"]
df_glm["DriverAgeClass"] = pd.cut(df_glm["DriverAge"], bins=bins_driver, labels=labels_driver)

bins_car = [-1, 1, 5, 10, 20, 200]
labels_car = ["0", "1-4", "5-9", "10-19", "20+"]
df_glm["CarAgeClass"] = pd.cut(df_glm["CarAge"], bins=bins_car, labels=labels_car)

bins_dens = [0, 50, 200, 500, 2000, 10000,np.inf]
labels_dens = ["rural", "peri-urbain", "petite ville", "ville", "urbain dense","urbain très dense"]
df_glm["DensityClass"] = pd.cut(df_glm["Density"], bins=bins_dens, labels=labels_dens)

# Déclaration des qualitatives
for col in ["Power", "Brand", "Gas", "Region",
            "DriverAgeClass", "CarAgeClass", "DensityClass"]:
    df_glm[col] = df_glm[col].astype("category")

print("Taille base GLM :", len(df_glm))


Taille base GLM : 413169


In [6]:
# =========================
# 2. Split 75 % / 25 %
# =========================

df_train, df_test = train_test_split(df_glm, test_size=0.25, random_state=123)

print("Taille train :", len(df_train))
print("Taille test  :", len(df_test))


Taille train : 309876
Taille test  : 103293


In [7]:
# Base des contrats sinistrés
df_gamma_sev_train = df_train[df_train["ClaimNb"] > 0].copy() #on garde que les contrats avec au moins un sinistre
df_gamma_sev_train["AvgClaim"] = df_gamma_sev_train["ClaimAmount"] / df_gamma_sev_train["ClaimNb"]
print("Taille base sévérité (train) :", len(df_gamma_sev_train))

df_gamma_sev_test = df_test[df_test["ClaimNb"] > 0].copy()
df_gamma_sev_test["AvgClaim"] = df_gamma_sev_test["ClaimAmount"] / df_gamma_sev_test["ClaimNb"]
print("Taille base sévérité (test) :", len(df_gamma_sev_test))

Taille base sévérité (train) : 11511
Taille base sévérité (test) : 3879


## Comparaison entre GLM Poisson et GLM Négative Binomiale Basé sur l'AIC et la deviance (pour la fréquence)

In [8]:
# =========================
# 3. Modèle Poisson complet
# =========================

formula_full = (
    "ClaimNb ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(Gas) + C(DensityClass) + C(Brand)"
)

model_full = smf.glm(
    formula=formula_full,
    data=df_train,
    family=sm.families.Poisson(),
    offset=np.log(df_train["Exposure"])
)
res_pois = model_full.fit()

aic_poi=res_pois.aic
dev_poi=res_pois.deviance
print("AIC Poisson complet :", res_pois.aic)
print("deviance",res_pois.deviance)


# =========================
# 3. Binomiale négative complète (train)
# =========================

nb_family = sm.families.NegativeBinomial()

model_nb = smf.glm(
    formula=formula_full,
    data=df_train,
    family=nb_family,
    offset=np.log(df_train["Exposure"])
)
res_nb = model_nb.fit()
aic_nb=res_nb.aic
dev_nb=res_nb.deviance
print("AIC Binomiale négative :", res_nb.aic)
print("deviance", res_nb.deviance)

AIC Poisson complet : 100940.90268717661
deviance 77490.80994923477


c:\Users\thoma\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


AIC Binomiale négative : 100688.9703756999
deviance 68102.04012360517


In [9]:
#Implémentation stepwise qui permet la sélection des variables selon l'AIC: une variable est ajoutée ou supprimée uniquement si cela diminue l’AIC du modèle courant ; sinon l’algorithme s’arrête.

def stepwise_glm_offset(data, response, candidates, family, offset_var, verbose=True):
    """
    data       : DataFrame
    response   : variable cible
    candidates : liste de termes de formule, ex. ["C(Region)", "C(DensityClass)"]
    family     : famille GLM, ex. sm.families.Poisson()
    offset_var : variable d'exposition
    """
    selected = []
    remaining = candidates.copy()
    current_aic = None
    changed = True

    while changed:
        changed = False

        # ----- FORWARD : ajout de la meilleure variable -----
        best_aic = None
        best_var = None

        for var in remaining:
            terms = selected + [var]
            rhs = " + ".join(terms) if terms else "1"
            formula = f"{response} ~ {rhs}"
            model = smf.glm(
                formula=formula,
                data=data,
                family=family,
                offset=np.log(data[offset_var])
            ).fit()
            aic = model.aic

            if (best_aic is None) or (aic < best_aic):
                best_aic = aic
                best_var = var

        if best_var is not None and (current_aic is None or best_aic < current_aic):
            selected.append(best_var)
            remaining.remove(best_var)
            current_aic = best_aic
            changed = True
            if verbose:
                print(f"Ajout de {best_var}, AIC = {current_aic:.2f}")

        # ----- BACKWARD : retrait de la pire variable -----
        improved = True
        while improved and len(selected) > 1:
            improved = False
            worst_aic = current_aic
            worst_var = None

            for var in selected:
                trial_vars = [v for v in selected if v != var]
                rhs = " + ".join(trial_vars) if trial_vars else "1"
                formula = f"{response} ~ {rhs}"
                model = smf.glm(
                    formula=formula,
                    data=data,
                    family=family,
                    offset=np.log(data[offset_var])
                ).fit()
                aic = model.aic

                if aic < worst_aic:
                    worst_aic = aic
                    worst_var = var

            if worst_var is not None:
                selected.remove(worst_var)
                current_aic = worst_aic
                improved = True
                changed = True
                if verbose:
                    print(f"Retrait de {worst_var}, AIC = {current_aic:.2f}")

    # ----- modèle final -----
    if selected:
        rhs = " + ".join(selected)
    else:
        rhs = "1"

    final_formula = f"{response} ~ {rhs}"
    final_model = smf.glm(
        formula=final_formula,
        data=data,
        family=family,
        offset=np.log(data[offset_var])
    ).fit()

    if verbose:
        print("Formule finale :", final_formula)

    return final_model, selected


In [10]:
df_poisson_freq_test2 = df_test.copy()
df_poisson_freq_train2 = df_train.copy()

candidates = [
    "C(DriverAgeClass)","C(CarAgeClass) ",
    "C(Power)","C(Region)","C(Gas)","(DensityClass)"
]

family = sm.families.Poisson()

model_step_poisson, vars_selected = stepwise_glm_offset(
    data=df_poisson_freq_train2,
    response="ClaimNb",
    candidates=candidates,
    family=family,
    offset_var="Exposure",
    verbose=True
)

print("Variables retenues :", vars_selected)
display(model_step_poisson.summary())


Ajout de C(DriverAgeClass), AIC = 101620.16
Ajout de (DensityClass), AIC = 101198.44
Ajout de C(Gas), AIC = 101076.79
Ajout de C(CarAgeClass) , AIC = 101038.00
Ajout de C(Power), AIC = 101006.19
Ajout de C(Region), AIC = 101005.71
Formule finale : ClaimNb ~ C(DriverAgeClass) + (DensityClass) + C(Gas) + C(CarAgeClass)  + C(Power) + C(Region)
Variables retenues : ['C(DriverAgeClass)', '(DensityClass)', 'C(Gas)', 'C(CarAgeClass) ', 'C(Power)', 'C(Region)']


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                ClaimNb   No. Observations:               309876
Model:                            GLM   Df Residuals:                   309840
Model Family:                 Poisson   Df Model:                           35
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -50467.
Date:                Tue, 09 Dec 2025   Deviance:                       77568.
Time:                        17:52:28   Pearson chi2:                 5.41e+05
No. Iterations:                     7   Pseudo R-squ. (CS):           0.004228
Covariance Type:            nonrobust                                         
=====================================================================================================
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -2.1969      0.061    -36.100      0.000      -2.316      -2.078
C(DriverAgeClass)[T.25-29]           -0.6078      0.041    -14.715      0.000      -0.689      -0.527
C(DriverAgeClass)[T.30-39]           -0.8585      0.035    -24.328      0.000      -0.928      -0.789
C(DriverAgeClass)[T.40-49]           -0.7098      0.034    -20.653      0.000      -0.777      -0.642
C(DriverAgeClass)[T.50-59]           -0.8126      0.036    -22.646      0.000      -0.883      -0.742
C(DriverAgeClass)[T.60+]             -0.8566      0.037    -22.904      0.000      -0.930      -0.783
DensityClass[T.peri-urbain]           0.1388      0.030      4.674      0.000       0.081       0.197
DensityClass[T.petite ville]          0.2544      0.033      7.781      0.000       0.190       0.319
DensityClass[T.ville]                 0.4030      0.031     12.949      0.000       0.342       0.464
DensityClass[T.urbain dense]          0.5443      0.035     15.491      0.000       0.475       0.613
DensityClass[T.urbain très dense]     0.6203      0.058     10.787      0.000       0.508       0.733
C(Gas)[T.Regular]                    -0.1815      0.020     -9.145      0.000      -0.220      -0.143
C(CarAgeClass)[T.1-4]                 0.0340      0.032      1.064      0.288      -0.029       0.097
C(CarAgeClass)[T.5-9]                 0.1192      0.032      3.726      0.000       0.056       0.182
C(CarAgeClass)[T.10-19]               0.0065      0.033      0.201      0.841      -0.057       0.070
C(CarAgeClass)[T.20+]                -0.3531      0.097     -3.656      0.000      -0.542      -0.164
C(Power)[T.e]                         0.1198      0.032      3.726      0.000       0.057       0.183
C(Power)[T.f]                         0.1069      0.032      3.361      0.001       0.045       0.169
C(Power)[T.g]                         0.0939      0.032      2.977      0.003       0.032       0.156
C(Power)[T.h]                         0.1505      0.045      3.377      0.001       0.063       0.238
C(Power)[T.i]                         0.2522      0.049      5.123      0.000       0.156       0.349
C(Power)[T.j]                         0.2347      0.049      4.750      0.000       0.138       0.332
C(Power)[T.k]                         0.2939      0.065      4.552      0.000       0.167       0.420
C(Power)[T.l]                         0.1912      0.093      2.055      0.040       0.009       0.374
C(Power)[T.m]                         0.2080      0.140      1.490      0.136      -0.066       0.482
C(Power)[T.n]                         0.2734      0.160      1.708      0.088      -0.040       0.587
C(Power)[T.o]                        -0.0166      0.182     -0.091      0.927      -0.372       0.339
C(Region)[T.Basse-Norma

In [11]:
aic_poisson=model_step_poisson.aic
dev_poisson=model_step_poisson.deviance
print("AIC Poisson :", aic_poisson)
print("deviance", dev_poisson)

AIC Poisson : 101005.70531891717
deviance 77567.61258097533


In [12]:
candidates = [
    "C(DriverAgeClass)", "C(CarAgeClass)",
    "C(Power)", "C(Region)", "C(Gas)", "C(DensityClass)"
]

# Famille binomiale négative au lieu de Poisson
family = sm.families.NegativeBinomial()

model_step_nb, vars_selected_nb = stepwise_glm_offset(
    data=df_poisson_freq_train2,
    response="ClaimNb",
    candidates=candidates,
    family=family,
    offset_var="Exposure",
    verbose=True
)

print("Variables retenues (NB) :", vars_selected_nb)
display(model_step_nb.summary())

c:\Users\thoma\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


Ajout de C(DriverAgeClass), AIC = 101330.05
Ajout de C(DensityClass), AIC = 100931.37
Ajout de C(Gas), AIC = 100816.24
Ajout de C(CarAgeClass), AIC = 100779.24
Ajout de C(Power), AIC = 100750.64
Formule finale : ClaimNb ~ C(DriverAgeClass) + C(DensityClass) + C(Gas) + C(CarAgeClass) + C(Power)
Variables retenues (NB) : ['C(DriverAgeClass)', 'C(DensityClass)', 'C(Gas)', 'C(CarAgeClass)', 'C(Power)']


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                ClaimNb   No. Observations:               309876
Model:                            GLM   Df Residuals:                   309849
Model Family:        NegativeBinomial   Df Model:                           26
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -50348.
Date:                Tue, 09 Dec 2025   Deviance:                       68194.
Time:                        17:53:50   Pearson chi2:                 5.26e+05
No. Iterations:                     7   Pseudo R-squ. (CS):           0.003994
Covariance Type:            nonrobust                                         
========================================================================================================
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
Intercept                               -2.2505      0.052    -43.410      0.000      -2.352      -2.149
C(DriverAgeClass)[T.25-29]              -0.6148      0.043    -14.414      0.000      -0.698      -0.531
C(DriverAgeClass)[T.30-39]              -0.8679      0.036    -23.803      0.000      -0.939      -0.796
C(DriverAgeClass)[T.40-49]              -0.7180      0.036    -20.198      0.000      -0.788      -0.648
C(DriverAgeClass)[T.50-59]              -0.8232      0.037    -22.204      0.000      -0.896      -0.751
C(DriverAgeClass)[T.60+]                -0.8695      0.039    -22.518      0.000      -0.945      -0.794
C(DensityClass)[T.peri-urbain]           0.1356      0.030      4.562      0.000       0.077       0.194
C(DensityClass)[T.petite ville]          0.2561      0.033      7.864      0.000       0.192       0.320
C(DensityClass)[T.ville]                 0.4081      0.030     13.420      0.000       0.348       0.468
C(DensityClass)[T.urbain dense]          0.5529      0.032     17.163      0.000       0.490       0.616
C(DensityClass)[T.urbain très dense]     0.6423      0.049     12.996      0.000       0.545       0.739
C(Gas)[T.Regular]                       -0.1823      0.020     -8.966      0.000      -0.222      -0.142
C(CarAgeClass)[T.1-4]                    0.0316      0.033      0.971      0.332      -0.032       0.095
C(CarAgeClass)[T.5-9]                    0.1166      0.032      3.601      0.000       0.053       0.180
C(CarAgeClass)[T.10-19]                  0.0044      0.033      0.134      0.894      -0.060       0.069
C(CarAgeClass)[T.20+]                   -0.3548      0.098     -3.621      0.000      -0.547      -0.163
C(Power)[T.e]                            0.1182      0.033      3.588      0.000       0.054       0.183
C(Power)[T.f]                            0.1044      0.033      3.206      0.001       0.041       0.168
C(Power)[T.g]                            0.0918      0.032      2.844      0.004       0.029       0.155
C(Power)[T.h]                            0.1494      0.046      3.271      0.001       0.060       0.239
C(Power)[T.i]                            0.2520      0.051      4.986      0.000       0.153       0.351
C(Power)[T.j]                            0.2359      0.051      4.650      0.000       0.136       0.335
C(Power)[T.k]                            0.2928      0.066      4.415      0.000       0.163       0.423
C(Power)[T.l]                            0.1910      0.096      2.000      0.045       0.004       0.378
C(Power)[T.m]                            0.2023      0.144      1.408      0.159      -0.079       0.484
C(Power)[T.n]                            0.2730      0.165      1.658      0.097      -0.050       0.596
C(Power)[T.o]                         

In [13]:
aic_nb=model_step_nb.aic
dev_nb=model_step_nb.deviance
print("AIC Poisson :", aic_nb)
print("deviance", dev_nb)

AIC Poisson : 100750.64284697578
deviance 68193.71259488109


In [14]:
# =========================
# 6. Modèle retenu
# =========================

if (model_step_nb.aic < model_step_poisson.aic) and (model_step_nb.deviance < model_step_poisson.deviance):
    res_best = model_step_nb
    best_name = "Binomiale négative"
else:
    res_best = model_step_poisson
    best_name = "Poisson"

print("Modèle retenu pour la tarification (comparaison Poisson vs NB) :", best_name)

df_best = df_test.copy()

# Fréquence prédite par contrat avec le modèle retenu
df_best["lambda_hat_best"] = res_best.predict(df_best, offset=np.log(df_best["Exposure"]))

freq_obs = df_best["ClaimNb"].sum() / df_best["Exposure"].sum()
freq_hat_nb = df_best["lambda_hat_best"].sum() / df_best["Exposure"].sum()
print(f"Fréquence observée : {freq_obs:.5f}")
print(f"Fréquence prédite (NB) : {freq_hat_nb:.5f}")

Modèle retenu pour la tarification (comparaison Poisson vs NB) : Binomiale négative
Fréquence observée : 0.07064
Fréquence prédite (NB) : 0.06987



### Modèle retenu pour la tarification : Binomiale négative

Le modèle de fréquence basé sur une distribution **Binomiale négative** a été retenu pour la tarification. Ce choix repose sur une comparaison avec le modèle Poisson, où la Binomiale négative s'est avérée plus adaptée pour gérer la sur-dispersion des données (variance supérieure à la moyenne).

- **Fréquence observée** : 0.07064  
    Cette valeur correspond à la fréquence moyenne des sinistres observés dans les données de test.

- **Fréquence prédite (NB)** : 0.06987  
    Cette valeur représente la fréquence moyenne des sinistres prédite par le modèle Binomiale négative. Bien que légèrement inférieure à la fréquence observée, elle reste cohérente avec les données et reflète la capacité du modèle à capturer les tendances globales.
```

## Comparaison entre GLM gamma et GLM lognormal basé sur l'AIC et la deviance 


In [15]:
#Implémentation stepwise qui permet la sélection des variables selon l'AIC: une variable est ajoutée ou supprimée uniquement si cela diminue l’AIC du modèle courant ; sinon l’algorithme s’arrête.

def stepwise_glm_offset_gamma(data, response, candidates, family, verbose=True):
    """
    data       : DataFrame
    response   : variable cible
    candidates : liste de termes de formule, ex. ["C(Region)", "C(DensityClass)"]
    family     : famille GLM, ex. sm.families.Poisson()
    offset_var : variable d'exposition
    """
    selected = []
    remaining = candidates.copy()
    current_aic = None
    changed = True

    while changed:
        changed = False

        # ----- FORWARD : ajout de la meilleure variable -----
        best_aic = None
        best_var = None

        for var in remaining:
            terms = selected + [var]
            rhs = " + ".join(terms) if terms else "1"
            formula = f"{response} ~ {rhs}"
            model = smf.glm(
                formula=formula,
                data=data,
                family=family
            ).fit()
            aic = model.aic

            if (best_aic is None) or (aic < best_aic):
                best_aic = aic
                best_var = var

        if best_var is not None and (current_aic is None or best_aic < current_aic):
            selected.append(best_var)
            remaining.remove(best_var)
            current_aic = best_aic
            changed = True
            if verbose:
                print(f"Ajout de {best_var}, AIC = {current_aic:.2f}")

        # ----- BACKWARD : retrait de la pire variable -----
        improved = True
        while improved and len(selected) > 1:
            improved = False
            worst_aic = current_aic
            worst_var = None

            for var in selected:
                trial_vars = [v for v in selected if v != var]
                rhs = " + ".join(trial_vars) if trial_vars else "1"
                formula = f"{response} ~ {rhs}"
                model = smf.glm(
                    formula=formula,
                    data=data,
                    family=family
                ).fit()
                aic = model.aic

                if aic < worst_aic:
                    worst_aic = aic
                    worst_var = var

            if worst_var is not None:
                selected.remove(worst_var)
                current_aic = worst_aic
                improved = True
                changed = True
                if verbose:
                    print(f"Retrait de {worst_var}, AIC = {current_aic:.2f}")

    # ----- modèle final -----
    if selected:
        rhs = " + ".join(selected)
    else:
        rhs = "1"

    final_formula = f"{response} ~ {rhs}"
    final_model = smf.glm(
        formula=final_formula,
        data=data,
        family=family
    ).fit()

    if verbose:
        print("Formule finale :", final_formula)

    return final_model, selected



In [16]:
df_sev_train_ln = df_gamma_sev_train.copy()
df_sev_test_ln  = df_gamma_sev_test.copy()



candidates_sev = [
    "C(DriverAgeClass)", "C(CarAgeClass)",
    "C(Power)", "C(Region)", "C(Gas)", "C(DensityClass)"
]

family_gamma = sm.families.Gamma(sm.families.links.log())

model_gamma, vars_gamma = stepwise_glm_offset_gamma(
    data=df_gamma_sev_train,        # DataFrame de sévérité (contrats sinistrés)
    response="AvgClaim",   # sévérité moyenne par sinistre ou par contrat
    candidates=candidates_sev,
    family=family_gamma,
    verbose=True
)


print("Variables Gamma retenues :", vars_gamma)
print("AIC Gamma :", model_gamma.aic)
display(model_gamma.summary())

c:\Users\thoma\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\genmod\families\links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


Ajout de C(DriverAgeClass), AIC = 238189.02
Ajout de C(Power), AIC = 231368.07
Ajout de C(Region), AIC = 230068.50
Ajout de C(CarAgeClass), AIC = 229748.88
Ajout de C(DensityClass), AIC = 229610.34
Ajout de C(Gas), AIC = 229584.96
Formule finale : AvgClaim ~ C(DriverAgeClass) + C(Power) + C(Region) + C(CarAgeClass) + C(DensityClass) + C(Gas)
Variables Gamma retenues : ['C(DriverAgeClass)', 'C(Power)', 'C(Region)', 'C(CarAgeClass)', 'C(DensityClass)', 'C(Gas)']
AIC Gamma : 229584.96408146693


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:               AvgClaim   No. Observations:                11511
Model:                            GLM   Df Residuals:                    11475
Model Family:                   Gamma   Df Model:                           35
Link Function:                    log   Scale:                          18.319
Method:                          IRLS   Log-Likelihood:            -1.1476e+05
Date:                Tue, 09 Dec 2025   Deviance:                       16637.
Time:                        17:54:02   Pearson chi2:                 2.10e+05
No. Iterations:                    45   Pseudo R-squ. (CS):           0.009066
Covariance Type:            nonrobust                                         
========================================================================================================
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
Intercept                                8.2198      0.269     30.588      0.000       7.693       8.747
C(DriverAgeClass)[T.25-29]              -0.7455      0.183     -4.068      0.000      -1.105      -0.386
C(DriverAgeClass)[T.30-39]              -0.6604      0.156     -4.221      0.000      -0.967      -0.354
C(DriverAgeClass)[T.40-49]              -0.8087      0.152     -5.319      0.000      -1.107      -0.511
C(DriverAgeClass)[T.50-59]              -0.8189      0.159     -5.147      0.000      -1.131      -0.507
C(DriverAgeClass)[T.60+]                -0.6355      0.166     -3.831      0.000      -0.961      -0.310
C(Power)[T.e]                            0.0325      0.142      0.229      0.819      -0.246       0.311
C(Power)[T.f]                            0.1359      0.140      0.970      0.332      -0.139       0.411
C(Power)[T.g]                            0.1496      0.138      1.082      0.279      -0.121       0.421
C(Power)[T.h]                           -0.1354      0.196     -0.691      0.489      -0.519       0.249
C(Power)[T.i]                            0.7209      0.215      3.345      0.001       0.299       1.143
C(Power)[T.j]                           -0.0160      0.217     -0.074      0.941      -0.442       0.410
C(Power)[T.k]                            0.0331      0.282      0.118      0.906      -0.519       0.585
C(Power)[T.l]                            0.2730      0.409      0.668      0.504      -0.528       1.074
C(Power)[T.m]                            0.6375      0.605      1.055      0.292      -0.547       1.822
C(Power)[T.n]                           -0.0935      0.712     -0.131      0.896      -1.490       1.303
C(Power)[T.o]                           -0.1679      0.790     -0.212      0.832      -1.717       1.381
C(Region)[T.Basse-Normandie]             0.2630      0.288      0.914      0.361      -0.301       0.827
C(Region)[T.Bretagne]                    0.0975      0.196      0.498      0.619      -0.286       0.481
C(Region)[T.Centre]                      0.0912      0.170      0.537      0.591      -0.242       0.424
C(Region)[T.Haute-Normandie]            -0.1717      0.387     -0.443      0.657      -0.931       0.587
C(Region)[T.Ile-de-France]               0.0277      0.200      0.138      0.890      -0.365       0.421
C(Region)[T.Limousin]                   -0.2007      0.409     -0.491      0.623      -1.002       0.601
C(Region)[T.Nord-Pas-de-Calais]         -0.0284      0.231     -0.123      0.902      -0.481       0.425
C(Region)[T.Pays-de-la-Loire]           -0.0935      0.202     -0.463      0.644      -0.490       0.303
C(Region)[T.Poitou-Charentes]           -0.1442      0.239     -0.602      0.547      -0.613       0.325
C(CarAgeClass)[T.1-4]                 

In [17]:
aic_g=model_gamma.aic
dev_g=model_gamma.deviance
print("AIC Gamma :", aic_g)
print("deviance", dev_g)

AIC Gamma : 229584.96408146693
deviance 16636.527593025075


In [18]:
def stepwise_ols(data, response, candidates, verbose=True):
    selected = []
    remaining = candidates.copy()
    current_aic = None
    changed = True

    while changed:
        changed = False

        # FORWARD
        best_aic = None
        best_var = None
        for var in remaining:
            terms = selected + [var]
            rhs = " + ".join(terms) if terms else "1"
            formula = f"{response} ~ {rhs}"
            model = smf.ols(formula=formula, data=data).fit()
            aic = model.aic
            if (best_aic is None) or (aic < best_aic):
                best_aic = aic
                best_var = var

        if best_var is not None and (current_aic is None or best_aic < current_aic):
            selected.append(best_var)
            remaining.remove(best_var)
            current_aic = best_aic
            changed = True
            if verbose:
                print(f"Ajout de {best_var}, AIC = {current_aic:.2f}")

        # BACKWARD
        improved = True
        while improved and len(selected) > 1:
            improved = False
            worst_aic = current_aic
            worst_var = None
            for var in selected:
                trial = [v for v in selected if v != var]
                rhs = " + ".join(trial) if trial else "1"
                formula = f"{response} ~ {rhs}"
                model = smf.ols(formula=formula, data=data).fit()
                aic = model.aic
                if aic < worst_aic:
                    worst_aic = aic
                    worst_var = var
            if worst_var is not None:
                selected.remove(worst_var)
                current_aic = worst_aic
                improved = True
                changed = True
                if verbose:
                    print(f"Retrait de {worst_var}, AIC = {current_aic:.2f}")

    rhs = " + ".join(selected) if selected else "1"
    final_formula = f"{response} ~ {rhs}"
    final_model = smf.ols(formula=final_formula, data=data).fit()
    if verbose:
        print("Formule finale lognormal :", final_formula)
    return final_model, selected



In [19]:
df_sev_train_ln["logAvg"] = np.log(df_sev_train_ln["AvgClaim"])
df_sev_test_ln["logAvg"]  = np.log(df_sev_test_ln["AvgClaim"])


candidates_sev = [
    "C(DriverAgeClass)", "C(CarAgeClass)",
    "C(Power)", "C(Region)", "C(Gas)", "C(DensityClass)"
]


model_logn, vars_logn = stepwise_ols(
    data=df_sev_train_ln,
    response="logAvg",
    candidates=candidates_sev,
    verbose=True
)

print("Variables lognormales retenues :", vars_logn)
print("AIC lognormal (train) :", model_logn.aic)
display(model_logn.summary())

Ajout de C(DriverAgeClass), AIC = 34816.50
Ajout de C(Region), AIC = 34814.46
Ajout de C(DensityClass), AIC = 34813.74
Formule finale lognormal : logAvg ~ C(DriverAgeClass) + C(Region) + C(DensityClass)
Variables lognormales retenues : ['C(DriverAgeClass)', 'C(Region)', 'C(DensityClass)']
AIC lognormal (train) : 34813.74215921269


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 logAvg   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     3.116
Date:                Tue, 09 Dec 2025   Prob (F-statistic):           5.38e-06
Time:                        17:54:03   Log-Likelihood:                -17387.
No. Observations:               11511   AIC:                         3.481e+04
Df Residuals:                   11491   BIC:                         3.496e+04
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
========================================================================================================
                                           coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
Intercept                                6.9997      0.057    123.522      0.000       6.889       7.111
C(DriverAgeClass)[T.25-29]              -0.1597      0.047     -3.413      0.001      -0.251      -0.068
C(DriverAgeClass)[T.30-39]              -0.1471      0.040     -3.698      0.000      -0.225      -0.069
C(DriverAgeClass)[T.40-49]              -0.0817      0.039     -2.111      0.035      -0.158      -0.006
C(DriverAgeClass)[T.50-59]              -0.0909      0.040     -2.250      0.024      -0.170      -0.012
C(DriverAgeClass)[T.60+]                 0.0068      0.042      0.161      0.872      -0.076       0.090
C(Region)[T.Basse-Normandie]            -0.0513      0.074     -0.696      0.486      -0.196       0.093
C(Region)[T.Bretagne]                   -0.0221      0.050     -0.442      0.659      -0.120       0.076
C(Region)[T.Centre]                     -0.1026      0.043     -2.366      0.018      -0.188      -0.018
C(Region)[T.Haute-Normandie]            -0.0229      0.099     -0.231      0.817      -0.217       0.171
C(Region)[T.Ile-de-France]               0.0533      0.051      1.039      0.299      -0.047       0.154
C(Region)[T.Limousin]                   -0.0370      0.105     -0.353      0.724      -0.242       0.168
C(Region)[T.Nord-Pas-de-Calais]         -0.0215      0.059     -0.364      0.716      -0.137       0.094
C(Region)[T.Pays-de-la-Loire]           -0.1023      0.052     -1.980      0.048      -0.203      -0.001
C(Region)[T.Poitou-Charentes]           -0.0716      0.061     -1.169      0.242      -0.192       0.048
C(DensityClass)[T.peri-urbain]          -0.0252      0.033     -0.758      0.448      -0.090       0.040
C(DensityClass)[T.petite ville]         -0.0512      0.036     -1.408      0.159      -0.123       0.020
C(DensityClass)[T.ville]                -0.0098      0.035     -0.283      0.777      -0.077       0.058
C(DensityClass)[T.urbain dense]         -0.0101      0.039     -0.260      0.795      -0.086       0.066
C(DensityClass)[T.urbain très dense]    -0.1764      0.065     -2.729      0.006      -0.303      -0.050
==============================================================================
Omnibus:                     1522.173   Durbin-Watson:                   1.997
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             6401.238
Skew:                          -0.600   Prob(JB):                         0.00
Kurtosis:                       6.451   Cond. No.                         17.2
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [20]:

# =========================
# 4. Déviance Gamma sur le test (critère commun)
# =========================
print("AIC Gamma :", aic_g)
print("AIC lognormal (train) :", model_logn.aic)

def gamma_deviance(mu, y):
    """Déviance Gamma (scale=1) pour vecteurs numpy positifs."""
    eps = 1e-10
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    mask = (y > 0) & (mu > 0) & ~np.isnan(mu)
    y = y[mask]
    mu = mu[mask]
    return 2 * np.sum((y - mu) / mu - np.log(y / mu + eps)), mask.sum()

# Prédictions Gamma
mu_gamma_test = model_gamma.predict(df_sev_test_ln)
dev_gamma, n_g = gamma_deviance(mu_gamma_test, df_sev_test_ln["AvgClaim"])
print(f"Déviance Gamma (test, n={n_g}) :", dev_gamma)

# Prédictions Lognormal ramenées au niveau moyen (E[Y] ≈ exp(m + s²/2])
mu_logn_test_ln = model_logn.predict(df_sev_test_ln)
sigma2 = model_logn.scale  # variance résiduelle sur log
mu_logn_test = np.exp(mu_logn_test_ln + 0.5 * sigma2)

dev_logn, n_l = gamma_deviance(mu_logn_test, df_sev_test_ln["AvgClaim"])
print(f"Déviance (critère Gamma) Lognormal (test, n={n_l}) :", dev_logn)

# =========================
# 5. Modèle de sévérité retenu
# =========================

if (model_logn.aic < model_gamma.aic) and (dev_logn < dev_gamma):
    sev_best = "Lognormal"
else:
    sev_best = "Gamma"

print("Modèle retenu pour la sévérité :", sev_best)


AIC Gamma : 229584.96408146693
AIC lognormal (train) : 34813.74215921269
Déviance Gamma (test, n=3879) : 6726.445478895816
Déviance (critère Gamma) Lognormal (test, n=3879) : 6573.558907028531
Modèle retenu pour la sévérité : Lognormal


### Choix du Modèle Lognormal pour la Sévérité

Après comparaison des performances des modèles Gamma et Lognormal pour la modélisation de la sévérité, le modèle **Lognormal** a été retenu. Voici les principales raisons de ce choix :

1. **AIC (Akaike Information Criterion)** :  
    - Le modèle Lognormal présente un AIC significativement plus faible que celui du modèle Gamma, indiquant une meilleure adéquation aux données d'entraînement.



2. **Performance globale** :  
    - Le modèle Lognormal capture mieux les relations dans les données, ce qui en fait un choix plus robuste pour la modélisation de la sévérité.

En conclusion, le modèle Lognormal est plus performant et sera utilisé pour estimer les coûts moyens conditionnels des sinistres.


In [21]:
res_sev_ln = model_logn #on recupere le modèle lognormal calculé plus tôt


print(res_sev_ln.summary())
print("AIC Lognormal sévérité (train) :", res_sev_ln.aic)

# Sévérité prédite (espérance lognormale) sur train / test
sigma2 = res_sev_ln.scale

df_sev_train_ln["logAvg_hat"] = res_sev_ln.predict(df_sev_train_ln)
df_sev_test_ln["logAvg_hat"]  = res_sev_ln.predict(df_sev_test_ln)

df_sev_train_ln["sev_hat"] = np.exp(df_sev_train_ln["logAvg_hat"] + 0.5 * sigma2)
df_sev_test_ln["sev_hat"]  = np.exp(df_sev_test_ln["logAvg_hat"]  + 0.5 * sigma2)



print(df_sev_test_ln["sev_hat"] .mean())

                            OLS Regression Results                            
Dep. Variable:                 logAvg   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     3.116
Date:                Tue, 09 Dec 2025   Prob (F-statistic):           5.38e-06
Time:                        17:54:03   Log-Likelihood:                -17387.
No. Observations:               11511   AIC:                         3.481e+04
Df Residuals:                   11491   BIC:                         3.496e+04
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
                                           coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------


In [22]:
#on rajoute les frequences estimées au dataframe de train et de test

df_train["lambda_hat"] = model_step_nb.predict(df_train, offset=np.log(df_train["Exposure"]))
df_test["lambda_hat"]  = model_step_nb.predict(df_test,  offset=np.log(df_test["Exposure"]))

# Rattacher sev_hat aux dataframes fréquence
df_train = df_train.merge(
    df_sev_train_ln[["PolicyID", "sev_hat"]],
    on="PolicyID",
    how="left"
)
df_test = df_test.merge(
    df_sev_test_ln[["PolicyID", "sev_hat"]],
    on="PolicyID",
    how="left"
)

# Tarification & validation

In [23]:
# Pour les contrats sans sinistre (pas de prédiction directe), on peut
# utiliser la sévérité moyenne globale estimée sur sev_train
sev_global = df_train["sev_hat"].mean()
df_train["sev_hat"] = df_train["sev_hat"].fillna(sev_global)
df_test["sev_hat"]  = df_test["sev_hat"].fillna(sev_global)

# ============================================================
# 4. Prime pure modélisée et tarif
# ============================================================

# Prime pure modélisée par contrat (unité d'exposition)
df_train["PurePremium_hat"] = df_train["lambda_hat"] * df_train["sev_hat"]
df_test["PurePremium_hat"]  = df_test["lambda_hat"]  * df_test["sev_hat"]

# Chargement global (ex : +20 %)
loading = 1.20
df_train["Tarif"] = df_train["PurePremium_hat"] * loading
df_test["Tarif"]  = df_test["PurePremium_hat"]  * loading

# ============================================================
# 5. Vérification du tarif sur l’échantillon de validation
#    -> comparaison prime pure observée vs modélisée par segment
# ============================================================

group_cols = ["DriverAgeClass", "Power", "Region"]

check_test = (
    df_test
    .groupby(group_cols)
    .agg(
        Exposure_tot=("Exposure", "sum"),
        Pure_obs=("PurePremium", "mean"),
        Pure_hat=("PurePremium_hat", "mean"),
        Tarif_moy=("Tarif", "mean")
    )
    .reset_index()
)

print("Vérification sur l'échantillon de validation (quelques segments) :")
display(check_test.sort_values("Exposure_tot", ascending=False).head(15))

# Comparaison globale sur le test
pure_obs_globale = (df_test["ClaimAmount"].sum() / df_test["Exposure"].sum())
pure_hat_globale = (df_test["PurePremium_hat"].sum() / df_test["Exposure"].sum())

print("Prime pure observée (test)  :", pure_obs_globale)
print("Prime pure modélisée (test) :", pure_hat_globale)

# Ratio modèle / observé
print("Ratio modèle / observé :", pure_hat_globale / pure_obs_globale)

Vérification sur l'échantillon de validation (quelques segments) :


C:\Users\thoma\AppData\Local\Temp\ipykernel_29996\1791397941.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(group_cols)


,DriverAgeClass,Power,Region,Exposure_tot,Pure_obs,Pure_hat,Tarif_moy
383,40-49,f,Centre,1608.721288,323.353863,75.487242,90.584691
263,30-39,f,Centre,1553.444164,305.425258,61.807606,74.169128
393,40-49,g,Centre,1473.794849,182.326372,70.105874,84.127049
623,60+,f,Centre,1446.532909,502.597644,73.944576,88.733492
513,50-59,g,Centre,1372.538418,214.761272,67.633788,81.160546
503,50-59,f,Centre,1306.773798,753.431430,69.165722,82.998866
633,60+,g,Centre,1272.075609,77.940247,68.336468,82.003762
273,30-39,g,Centre,1265.293205,148.026016,54.913754,65.896505
373,40-49,e,Centre,1123.480222,99.148069,76.253510,91.504212
253,30-39,e,Centre,1116.862079,1096.095959,64.110241,76.932289


Prime pure observée (test)  : 148.27044660402692
Prime pure modélisée (test) : 119.26003560457244
Ratio modèle / observé : 0.8043412449081638


## mise en place d'un GLM poisson avec lasso pour la selection de variables


### GLM Lasso

In [24]:
import numpy as np
# compat NumPy
np.float = float

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from pyglmnet import GLMCV

# =========================
# 1. Préparation des données
# =========================

df = df_glm.copy()
df = df[df["Exposure"] > 0].copy()

# Réponse = nombre de sinistres (on passe l'expo en offset)
y = df["ClaimNb"].to_numpy()

# Variables explicatives (toutes qualitatives)
X_cat = df[["DriverAgeClass", "CarAgeClass", "Power",
            "Region", "Gas", "DensityClass"]].astype("category")

enc = OneHotEncoder(drop=None, sparse_output=False)
X_encoded = enc.fit_transform(X_cat)


# Split 75 / 25 pour évaluer ensuite
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.25, random_state=123
)

X_train = np.asarray(X_train, dtype=np.float64)
y_train = np.asarray(y_train, dtype=np.float64)

print(X_train.dtype)
print(y_train.shape)

# =========================
# 2. GLMCV : Poisson + Lasso
# =========================

# grille de lambda (équivalent à ton np.linspace, mais GLMCV attend un array) [web:21]
lambda_path = np.linspace(0, 0.1, 5)

glm_cv = GLMCV(
    distr="poisson",
    alpha=1.0,                  # Lasso pur (L1) [web:24]     
    cv=5,                       # 5-fold CV
    score_metric="pseudo_R2",   # comme dans ton code [web:24][web:15]
    max_iter=200,
    learning_rate=1e-3,
    tol=1e-5,
    solver="cdfast",            # identique à ton GLM
    verbose=2
)

glm_cv.fit(X_train, y_train)



Looping through the regularization path
Lambda: 0.5000
Lambda: 0.5000


float64
(309876,)


	Parameter update tolerance. Converged in 7 iterations
Lambda: 0.5000
	Parameter update tolerance. Converged in 7 iterations
Lambda: 0.5000
	Parameter update tolerance. Converged in 7 iterations
Lambda: 0.5000
	Parameter update tolerance. Converged in 7 iterations
Lambda: 0.5000
	Parameter update tolerance. Converged in 7 iterations
Lambda: 0.5000
	Parameter update tolerance. Converged in 7 iterations
Lambda: 0.3237
Lambda: 0.3237
	Parameter update tolerance. Converged in 9 iterations
Lambda: 0.3237
	Parameter update tolerance. Converged in 9 iterations
Lambda: 0.3237
	Parameter update tolerance. Converged in 9 iterations
Lambda: 0.3237
	Parameter update tolerance. Converged in 8 iterations
Lambda: 0.3237
	Parameter update tolerance. Converged in 9 iterations
Lambda: 0.3237
	Parameter update tolerance. Converged in 9 iterations
Lambda: 0.2096
Lambda: 0.2096
	Parameter update tolerance. Converged in 8 iterations
Lambda: 0.2096
	Parameter update tolerance. Converged in 9 iterations
Lambd

<
Distribution | poisson
alpha | 1.00
max_iter | 200.00
lambda: 0.50 to 0.01
>

In [25]:
# =========================
# 3. Lambda optimal et modèle final
# =========================

# lambda choisi par la CV (indice de la meilleure valeur)
print("Lambda choisi par la CV :", glm_cv.reg_lambda_opt_)

# Coefficients associés au meilleur lambda
beta0 = glm_cv.beta0_
coef = glm_cv.beta_

print("Intercept :", beta0)
print("Nb de coefficients non nuls :", np.count_nonzero(coef))

feature_names = enc.get_feature_names_out(X_cat.columns)
selected = [(name, c) for name, c in zip(feature_names, coef) if abs(c) > 1e-6]
selected = sorted(selected, key=lambda x: x[1], reverse=True)

#print("Variables sélectionnées par le Lasso (coefs non nuls) :")
#for name, c in selected:
#    print(f"{name:35s}  coef={c:.4f}")

print("variables nulles:")
for name, c in zip(feature_names, coef):
    if abs(c) <= 1e-6:
        print(f"{name:35s}  coef={c:.4f}")

# =========================
# 4. Évaluation sur le test (optionnel)
# =========================

from pyglmnet import metrics

lasso_pred = glm_cv.predict(X_test)
deviance_test = metrics.deviance(y_test, lasso_pred, distr="poisson")  # [web:24]
print(f"Deviance (test) : {deviance_test:.4f}")


# Nombre de paramètres estimés (coefficients non nuls + intercept)
k = np.count_nonzero(glm_cv.beta_) + 1  # +1 pour l'intercept

# Calcul de la log-vraisemblance
log_likelihood = -metrics.deviance(y_test, glm_cv.predict(X_test), distr="poisson") / 2

# Calcul de l'AIC
aic_glmcv = 2 * k - 2 * log_likelihood

print("Nombre de paramètres estimés :", k)
print("Log-vraisemblance :", log_likelihood)
print("AIC pour GLMCV :", aic_glmcv)


# 4) Fréquence prédite et contrôle global
#lasso_pred = model_lasso.predict(X_test)

freq_obs = df_test["ClaimNb"].sum() / df_test["Exposure"].sum()
freq_hat_nb = lasso_pred.sum() / df_test["Exposure"].sum()
print(f"Fréquence observée : {freq_obs:.5f}")
print(f"Fréquence prédite (lasso) : {freq_hat_nb:.5f}")


Lambda choisi par la CV : 0.015444521049463802
Intercept : -3.0474383236927234
Nb de coefficients non nuls : 29
variables nulles:
DriverAgeClass_50-59                 coef=-0.0000
CarAgeClass_1-4                      coef=-0.0000
CarAgeClass_10-19                    coef=-0.0000
Power_f                              coef=-0.0000
Power_h                              coef=-0.0000
Power_m                              coef=-0.0000
Power_n                              coef=-0.0000
Region_Basse-Normandie               coef=-0.0000
Region_Centre                        coef=-0.0000
Region_Pays-de-la-Loire              coef=-0.0000
Gas_Diesel                           coef=-0.0000
DensityClass_ville                   coef=-0.0000
Deviance (test) : 26849.5827
Nombre de paramètres estimés : 30
Log-vraisemblance : -13424.79134707864
AIC pour GLMCV : 26909.58269415728
Fréquence observée : 0.07064
Fréquence prédite (lasso) : 0.06947


Sur les 41 variables indicatrices issues du codage des facteurs, 35 coefficients seulement sont non nuls, ce qui signifie que le Lasso a effectué une sélection effective de variables en annulant plusieurs modalités jugées peu informatives, comme certaines classes d’âge du véhicule, de puissance ou de densité. Les modalités listées avec un coefficient nul (par exemple CarAgeClass_1-4, Region_Centre, Gas_Diesel, etc.) n’apportent pas de contribution supplémentaire à la fréquence de sinistre par rapport au profil de référence, au regard de la pénalisation retenue.

La fréquence observée des sinistres sur le test est de 0,07064, tandis que la fréquence moyenne prédite par le modèle Lasso est de 0,06946 : l’alignement très proche entre fréquence observée et prédite montre que le modèle reproduit correctement le niveau moyen de risque, tout en restant parcimonieux grâce à la régularisation.

In [26]:
#on rajoute les frequences estimées au dataframe de train et de test

df_train["lambda_hat_cv"] = glm_cv.predict(X_train)
df_test["lambda_hat_cv"]  = glm_cv.predict(X_test)


### resultats du  GLM lasso

In [27]:

# Prime pure modélisée par contrat (unité d'exposition)
df_train["PurePremium_hat_cv"] = df_train["lambda_hat_cv"] * df_train["sev_hat"]
df_test["PurePremium_hat_cv"]  = df_test["lambda_hat_cv"]  * df_test["sev_hat"]

# Chargement global (ex : +20 %)
loading = 1.20
df_train["Tarif_cv"] = df_train["PurePremium_hat_cv"] * loading
df_test["Tarif_cv"]  = df_test["PurePremium_hat_cv"]  * loading

# Comparaison globale sur le test
pure_obs_globale_cv = (df_test["ClaimAmount"].sum() / df_test["Exposure"].sum())
pure_hat_globale_cv = (df_test["PurePremium_hat_cv"].sum() / df_test["Exposure"].sum())

print("Prime pure observée (test)  :", pure_obs_globale_cv)
print("Prime pure modélisée (test) :", pure_hat_globale_cv)

# Ratio modèle / observé
print("Ratio modèle / observé :", pure_hat_globale_cv / pure_obs_globale_cv)

Prime pure observée (test)  : 148.27044660402692
Prime pure modélisée (test) : 118.55565488050911
Ratio modèle / observé : 0.799590596750042
